# REST API — FastAPI

Run this cell to start the server, then send requests to `http://localhost:8000/predict`

In [ ]:
import joblib
import pandas as pd
import numpy as np
from pydantic import BaseModel
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from fastapi.staticfiles import StaticFiles
import uvicorn
import threading
from pathlib import Path

model = joblib.load('../models/churn_model.pkl')
scaler = joblib.load('../models/scaler.pkl')

# Recreate column order from training
X_train = pd.read_parquet('../data/X_train.parquet')
feature_cols = X_train.columns.tolist()

app = FastAPI(title='Churn Prediction API', version='1.0.0')

static_dir = Path('../static').resolve()
if static_dir.exists():
    app.mount('/static', StaticFiles(directory=str(static_dir)), name='static')

@app.get('/')
def index():
    return FileResponse('../templates/index.html')

In [ ]:
class CustomerInput(BaseModel):
    gender: str
    SeniorCitizen: int
    Partner: str
    Dependents: str
    tenure: int
    PhoneService: str
    MultipleLines: str
    InternetService: str
    OnlineSecurity: str
    OnlineBackup: str
    DeviceProtection: str
    TechSupport: str
    StreamingTV: str
    StreamingMovies: str
    Contract: str
    PaperlessBilling: str
    PaymentMethod: str
    MonthlyCharges: float
    TotalCharges: float

class PredictionOut(BaseModel):
    churn: int
    probability: float

In [ ]:
@app.post('/predict', response_model=PredictionOut)
def predict(data: CustomerInput):
    try:
        row = pd.DataFrame([data.model_dump()])

        row['TotalCharges'] = pd.to_numeric(row['TotalCharges'], errors='coerce').fillna(0)

        binary_map = {'Yes': 1, 'No': 0}
        for c in ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
            row[c] = row[c].map(binary_map)
        row['gender'] = (row['gender'] == 'Male').astype(int)

        cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity',
                    'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
                    'StreamingMovies', 'Contract', 'PaymentMethod']
        row = pd.get_dummies(row, columns=cat_cols, drop_first=True, dtype=int)

        # Align to training columns — add missing, drop extra
        for c in feature_cols:
            if c not in row.columns:
                row[c] = 0
        row = row[feature_cols]

        num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
        row[num_cols] = scaler.transform(row[num_cols])

        prob = model.predict_proba(row)[0, 1]
        pred = int(prob >= 0.5)
        return PredictionOut(churn=pred, probability=round(prob, 4))
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

In [ ]:
# Health check
@app.get('/health')
def health():
    return {'status': 'ok'}

### Start server

Run this cell to start the server in the background on port 8000.

In [ ]:
threading.Thread(target=uvicorn.run, args=(app,), kwargs={'host': '0.0.0.0', 'port': 8000}, daemon=True).start()
print('Server started at http://localhost:8000')
print('API docs at http://localhost:8000/docs')

### Test prediction

In [ ]:
import httpx

sample = {
    'gender': 'Male',
    'SeniorCitizen': 0,
    'Partner': 'Yes',
    'Dependents': 'No',
    'tenure': 12,
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'Yes',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'Yes',
    'StreamingMovies': 'Yes',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 70.0,
    'TotalCharges': 840.0,
}

resp = httpx.post('http://localhost:8000/predict', json=sample, timeout=10)
print(resp.status_code, resp.json())